In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from pathlib import Path
from tqdm.auto import tqdm

g:\NeuroMed-VLM\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
checkpoint_path = Path("checkpoints/day7_projection.pt")

print("Current directory:", Path.cwd())
print("Checkpoint exists:", checkpoint_path.exists())
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print(checkpoint.keys())

Current directory: g:\NeuroMed-VLM
Checkpoint exists: True
dict_keys(['image_projection', 'text_projection', 'optimizer', 'temperature'])


In [4]:
image_projection = nn.Linear(384, 256)
text_projection = nn.Linear(768, 256)

In [5]:
image_projection.load_state_dict(
    checkpoint["image_projection"]
)

text_projection.load_state_dict(
    checkpoint["text_projection"]
)

<All keys matched successfully>

In [6]:
for name, param in image_projection.state_dict().items():
    saved = checkpoint["image_projection"][name]
    print(name, torch.equal(param, saved))

for name, param in text_projection.state_dict().items():
    saved = checkpoint["text_projection"][name]
    print(name, torch.equal(param, saved))

weight True
bias True
weight True
bias True


In [7]:
optimizer = torch.optim.AdamW(
    list(image_projection.parameters()) +
    list(text_projection.parameters()),
    lr=1e-4
)

optimizer.load_state_dict(checkpoint["optimizer"])

In [8]:
for param, state in optimizer.state.items():
    print("Step:", state["step"].item())
    print("exp_avg shape:", state["exp_avg"].shape)
    print("exp_avg_sq shape:", state["exp_avg_sq"].shape)
    break

Step: 2500.0
exp_avg shape: torch.Size([256, 384])
exp_avg_sq shape: torch.Size([256, 384])


In [24]:
from src.train import train_one_epoch
from src.dataset import MedicalDataset, collate_fn, create_dataloader

print("Dataset module imported successfully!")

Dataset module imported successfully!


In [25]:
from datasets import load_dataset
from itertools import islice

ds = load_dataset(
    "eltorio/ROCOv2-radiology",
    split="train",
    streaming=True
)

samples = list(islice(
    ds.shuffle(seed=42, buffer_size=5000),
    1000
))

print("Training samples:", len(samples))

Training samples: 1000


In [26]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [27]:
from transformers import AutoTokenizer
from src.dataset import MedicalDataset, create_dataloader

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

train_dataset = MedicalDataset(samples, transform=transform)

loader = create_dataloader(
    train_dataset,
    tokenizer,
    batch_size=4,
    shuffle=True
)

In [28]:
batch = next(iter(loader))

print("Images:", batch["pixel_values"].shape)
print("Input IDs:", batch["input_ids"].shape)
print("Attention Mask:", batch["attention_mask"].shape)
print("Image IDs:", len(batch["image_ids"]))
print("Number of batches:", len(loader))

Images: torch.Size([4, 3, 224, 224])
Input IDs: torch.Size([4, 78])
Attention Mask: torch.Size([4, 78])
Image IDs: 4
Number of batches: 250


In [29]:
test_loader = [batch]

print("Test batches:", len(test_loader))

Test batches: 1


In [31]:
from transformers import AutoModel

image_encoder = AutoModel.from_pretrained(
    "facebook/dinov2-small"
)

text_encoder = AutoModel.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

for param in image_encoder.parameters():
    param.requires_grad = False

for param in text_encoder.parameters():
    param.requires_grad = False

image_encoder.eval()
text_encoder.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15348.78it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [32]:
import copy

test_image_projection = copy.deepcopy(image_projection)
test_text_projection = copy.deepcopy(text_projection)

test_optimizer = torch.optim.AdamW(
    list(test_image_projection.parameters()) +
    list(test_text_projection.parameters()),
    lr=1e-4
)

test_optimizer.load_state_dict(
    copy.deepcopy(optimizer.state_dict())
)

In [33]:
from src.train import train_one_epoch

test_loss = train_one_epoch(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    image_projection=test_image_projection,
    text_projection=test_text_projection,
    loader=test_loader,
    optimizer=test_optimizer,
    temperature=checkpoint["temperature"]
)

print("Test Epoch Loss:", test_loss)

Training: 100%|██████████| 1/1 [00:00<00:00,  2.22it/s, avg_loss=0.1152, batch_loss=0.1152]

Test Epoch Loss: 0.11519479751586914


In [34]:
print(
    "Original image unchanged:",
    torch.equal(
        image_projection.weight,
        checkpoint["image_projection"]["weight"]
    )
)

print(
    "Original text unchanged:",
    torch.equal(
        text_projection.weight,
        checkpoint["text_projection"]["weight"]
    )
)

print(
    "Test image updated:",
    not torch.equal(
        test_image_projection.weight,
        image_projection.weight
    )
)

print(
    "Test text updated:",
    not torch.equal(
        test_text_projection.weight,
        text_projection.weight
    )
)

Original image unchanged: True
Original text unchanged: True
Test image updated: True
Test text updated: True


In [35]:
from src.evaluate import evaluate_retrieval

image_embeddings = torch.eye(10)
text_embeddings = torch.eye(10)

results = evaluate_retrieval(image_embeddings, text_embeddings)

print(results)

Similarity: torch.Size([10, 10])
{'I2T_R@1': 1.0, 'T2I_R@1': 1.0, 'I2T_R@5': 1.0, 'T2I_R@5': 1.0, 'I2T_R@10': 1.0, 'T2I_R@10': 1.0}
